# Songo Lot 40 — MCTS CPU Pipeline Optimization
Exécuter les stages dans l'ordre. Chaque résultat est persistant sur Drive.

In [ ]:
from pathlib import Path
import os,subprocess,sys,time
REPO_DIR=Path('/content/songo-fish'); REPO_URL='https://github.com/GlennEriss/songo-fish.git'
subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True) if not REPO_DIR.exists() else subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
os.chdir(REPO_DIR); subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[train,perf,dev]'],check=True); os.environ['PYTHONPATH']='packages:apps/trainer/scripts'


In [ ]:
from google.colab import drive,files
drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/songo-ai'); INPUT=ROOT/'inputs/lot40_inputs.tar.gz'; OUTPUT=ROOT/'experiments/lot40_mcts_cpu_pipeline'; EXPORT=ROOT/'exports/lot40_results.tar.gz'
subprocess.run(['tar','-xzf',str(INPUT),'-C',str(REPO_DIR)],check=True); OUTPUT.mkdir(parents=True,exist_ok=True)
RUN=[sys.executable,'-u','apps/trainer/scripts/run_srn_lot40.py']
def supervised_run(cmd,label):
 log=Path('/content')/f'{label}.log'; stream=log.open('wb'); p=subprocess.Popen(cmd,stdout=stream,stderr=subprocess.STDOUT,cwd=REPO_DIR); start=time.monotonic(); off=0
 while p.poll() is None:
  time.sleep(10); data=log.read_bytes()[off:]
  if data: print(data.decode(errors='replace'),end='',flush=True); off+=len(data)
  print(f'[{label}] actif depuis {time.monotonic()-start:.0f}s',flush=True)
 stream.close(); data=log.read_bytes()[off:]
 if data: print(data.decode(errors='replace'),end='',flush=True)
 if p.returncode: raise RuntimeError(f'{label} code={p.returncode}; log={log}')


In [ ]:
supervised_run(RUN+['--stage','prepare','--device','cuda','--positions-file','data/colab_bridge/lot39_benchmark_positions.json','--lot39-reference','data/colab_bridge/lot40_reference/lot39_scaling_256.json','--output',str(OUTPUT)],'lot40_prepare')

In [ ]:
supervised_run(RUN+['--stage','00_baseline','--device','cuda','--output',str(OUTPUT)],'lot40_baseline')

In [ ]:
supervised_run(RUN+['--stage','01_profile','--device','cuda','--output',str(OUTPUT)],'lot40_profile')

In [ ]:
supervised_run(RUN+['--stage','02_coordination','--device','cuda','--output',str(OUTPUT)],'lot40_o1')

In [ ]:
supervised_run(RUN+['--stage','03_tree','--device','cuda','--output',str(OUTPUT)],'lot40_o2')

In [ ]:
supervised_run(RUN+['--stage','04_engine','--device','cuda','--output',str(OUTPUT)],'lot40_o3')

In [ ]:
supervised_run(RUN+['--stage','05_graph','--device','cuda','--output',str(OUTPUT)],'lot40_o4')

In [ ]:
supervised_run(RUN+['--stage','06_final','--device','cuda','--output',str(OUTPUT)],'lot40_final'); supervised_run(RUN+['--stage','07_single_search','--device','cuda','--output',str(OUTPUT)],'lot40_single')

In [ ]:
supervised_run(RUN+['--stage','08_finalize','--output',str(OUTPUT)],'lot40_finalize'); supervised_run(RUN+['--stage','export','--output',str(OUTPUT),'--bundle',str(EXPORT)],'lot40_export')